# Speech Transcription with Whisper

[Whisper](https://github.com/openai/whisper) is OpenAI's speech-to-text model, capable of multilingual transcription and translation. We use [faster-whisper](https://github.com/SYSTRAN/faster-whisper), a CTranslate2-based reimplementation that offers up to 4× faster inference with lower memory usage.

In this notebook, we record audio directly in the browser and compare transcription quality across different model sizes.

## Setup

In [ ]:
from faster_whisper import WhisperModel
from ipywebrtc import CameraStream, AudioRecorder
from IPython.display import Audio, Markdown, display
from ai_dojo import show
import tempfile

## Record Audio

We use [ipywebrtc](https://github.com/maartenbreddels/ipywebrtc) to capture audio from your microphone via the browser's WebRTC API.

**Instructions:**
1. Grant microphone access when prompted
2. Click the **record button** (red circle) to start
3. Click again to stop

In [ ]:
# Create an audio-only stream from the microphone
stream = CameraStream(constraints={"audio": True, "video": False})

# Create the recorder widget
recorder = AudioRecorder(stream=stream)
recorder

After recording, run the cell below to save and play back the audio.

In [ ]:
# Save recorded audio to a temporary file
temp_file = tempfile.NamedTemporaryFile(suffix=".webm", delete=False)
temp_file.write(recorder.audio.value)
temp_file.close()

audio_path = temp_file.name
print(f"Saved {len(recorder.audio.value):,} bytes")

# Play back the recording
Audio(audio_path)

### Alternative: Upload Audio File

If microphone recording is not available (e.g., on a remote server), you can upload an audio file instead.

In [ ]:
import ipywidgets as widgets

uploader = widgets.FileUpload(
    accept=".wav,.mp3,.webm,.m4a,.ogg,.flac",
    multiple=False,
    description="Upload Audio"
)
uploader

In [ ]:
# Run this cell after uploading a file
if uploader.value:
    uploaded_file = uploader.value[0]  # ipywidgets 8.x returns a tuple
    
    suffix = "." + uploaded_file.name.split(".")[-1]
    temp_file = tempfile.NamedTemporaryFile(suffix=suffix, delete=False)
    temp_file.write(uploaded_file.content)
    temp_file.close()
    
    audio_path = temp_file.name
    print(f"Uploaded: {uploaded_file.name} ({len(uploaded_file.content):,} bytes)")
    
    Audio(audio_path)
else:
    print("No file uploaded yet. Upload a file above, then re-run this cell.")

## Transcribe with Whisper

Whisper comes in different model sizes — larger models are more accurate but slower:

| Model | Parameters | Relative Speed | Use Case |
|-------|------------|----------------|----------|
| `tiny` | 39M | ~10× | Quick tests, low resources |
| `base` | 74M | ~7× | Fast, decent quality |
| `small` | 244M | ~4× | Good balance |
| `medium` | 769M | ~2× | High quality |
| `large-v3` | 1550M | 1× | Best quality |

Models are downloaded automatically on first use.

In [ ]:
def transcribe(audio_path: str, model_size: str = "base") -> str:
    """Transcribe audio using faster-whisper."""
    model = WhisperModel(model_size, device="cpu", compute_type="int8")
    
    segments, info = model.transcribe(audio_path, beam_size=5)
    
    display(Markdown(f"**Detected language:** {info.language} ({info.language_probability:.0%} confidence)"))
    
    text = " ".join(segment.text for segment in segments)
    return text.strip()

### Tiny Model

In [ ]:
%%time
text_tiny = transcribe(audio_path, model_size="tiny")
show.text(text_tiny)

### Base Model

In [ ]:
%%time
text_base = transcribe(audio_path, model_size="base")
show.text(text_base)

### Small Model

In [ ]:
%%time
text_small = transcribe(audio_path, model_size="small")
show.text(text_small)

## Comparison

All results side by side:

In [ ]:
import pandas

comparison = pandas.DataFrame({
    "Model": ["tiny", "base", "small"],
    "Transcription": [text_tiny, text_base, text_small]
})

show.dataframe_with_text(comparison, max_colwidth=600, max_chars=2000)

---
_This notebook is licensed under a [Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International (CC BY-NC-SA 4.0)](https://creativecommons.org/licenses/by-nc-sa/4.0/). Copyright © 2025-2026 [Christian Staudt](https://clstaudt.me)_